## Publish, Build, and Manage a Dynamic Map Service Cache
This notebook demonstrates a publishing workflow for Arcgis Enterprise. It illustrates publishing a map service, creating its cache schema, building selected tiles, and managing operations for the cache. Please see the [Map Caching](https://doc.esri.com/en/arcgis-enterprise/latest/administer/what-is-map-caching-.html?pivots=os-windows) and  [Caching terminology](https://doc.esri.com/en/arcgis-enterprise/latest/administer/caching-terminology.html?pivots=os-windows) documentation for detailed conceptual information and explanations. 

### 1. Connect to ArcGIS Enterprise
Load the required libraries and create the portal connection used throughout the notebook.

In [5]:
from pathlib import Path
from arcgis.gis import GIS
from arcgis.layers import MapImageLayer
from arcgis.geometry import Polygon
from arcgis.geometry import Envelope

#Connect to GIS
#User can create the profile for standalone server, Enterprise portal or Kubernetes
gis = GIS(profile='Home')

### 2. Publish the Service Definition file
The code below assigns a variable to the location of a service definition file. See the [Publish service definition files](https://doc.esri.com/en/arcgis-enterprise/latest/administer/about-service-definition-files.html?pivots=os-windows#6D4) documentation for information about service definition files. It then assigns a variable to the designated server on which to run the API's [publish_sd()](https://developers.arcgis.com/python/latest/api-reference/arcgis.gis.server.html#arcgis.gis.server.Server.publish_sd) operation to publish a Map Image Layer item.

> **Note:** You may see warnings about location differences in the service definition file and the ArcGIS Server.

In [9]:
#retrieve the sd file from the data location

data_path = Path(r"\\Location\to\the\SDFile.sd")

#Get the Hosted/Federated Server
#server = gis.admin.servers.get(role=FEDERATED_SERVER")[0]
server = gis.admin.servers.get(role="HOSTING_SERVER")[0]

#Publishing the SD
publish_result = server.publish_sd( sd_file=str(data_path))
#  folder="Optional: Folder Name to which you wish to publish" )

WARNING 086222: Output directory in service definition is not set or is invalid. Using default output directory.


#### Get the Map Image Layer item and initialize a MapImageLayer object from it

In [ ]:
#Search for the item that was just published
msvc_item = gis.content.search("<NameOfTheService*>", "Map Service")[0]

#Getting the Map Image Layer
mimg_lyr = MapImageLayer.fromitem(item=msvc_item)

### 3. Create cache schema
The code in the cell below runs the [create_cache_schema()](/python/latest/api-reference/arcgis.layers.toc.html#arcgis.layers.EnterpriseMapImageLayerManager.create_cache_schema) method to define the tiling scheme and preparatory folders, known as the cache schema for the Map Service. The operation does not generate tiles for the cache. The ArcGIS API for Python tool calls the _CachingTools_ geoprocessing service on the ArcGIS Server component of the enterprise deployment, and runs the [Create Map Server Cache](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/server/create-map-server-cache.html?tabs=dialog) task to process the request. Click the link to the documentation for detailed explanations on parameters.

> **Note:** This step can be skipped if the map is already cached.

In [ ]:
#Defining variables for creating cache schema

#Cache directory of the Enterprise
cache_dir = "\\Location\to\the\arcgiscache\directory\of\the\server"

# Scales of the tiling schema that will be used. AGOL Tiling Scheme is used in this example.
scales = [591657527.591555, 295828763.79577702, 147914381.89788899, 73957190.948944002, 36978595.474472001, 18489297.737236001,
        9244648.8686180003, 4622324.4343090001, 2311162.2171550002, 1155581.108577, 577790.55428899999, 288895.27714399999, 
        144447.638572, 72223.819285999998, 36111.909642999999, 18055.954822, 9027.9774109999998, 4513.9887049999998, 2256.994353,
        1128.4971760000001, 564.24858800000004, 282.12429400000002, 141.06214700000001, 70.531073500000005 ]

# The upper-left origin of the tiling scheme in map units - as space separated values.
tile_origin_x_y = "-20037508.342787 20037508.342787"

#Cache Format Specifies the cache tile format that will be used (PNG(8,24,32) , JPEG, Mixed)
cache_format= 'JPEG'

#In case of JPEG image format the default value is 75
tile_comp_quality = 75

#Retrieve the EnterpriseMapImageLayerManager object for the MapImageLayer
ent_mapimg_mgr = mimg_lyr.manager

#Storage Format can be either COMPACT or COMPACT_V2
storage_format= 'COMPACT_V2'

#The width and height of the cache tiles. This is in pixels. It can be 128 x 128, 256 x 256, 512 x 512 or 1024 x 1024
tile_width = 256
tile_height = 256


# The dots per inch (DPI) of the intended output device, default value is 96
dpi = 96

#Creating the Cache Schema for the map Service
ent_mapimg_mgr.create_cache_schema( 
   cache_directory= cache_dir,
   levels= scales ,
   tile_origin= tile_origin_x_y,
   storage_format= storage_format,
   cache_format= cache_format,
   tile_compression_quality= tile_comp_quality,
   dpi= dpi,
   tile_width = tile_width,
   tile_height = tile_height,
   use_local_cache_dir = True,
   lerc_error =0,
   ready_to_serve_format = True
)

### 5. Generate tiles for the selected scales and area
This code generates the cache for the specified scales and a specific area of interest using the API for Python's [build_cache()](/python/latest/api-reference/arcgis.layers.toc.html#arcgis.layers.EnterpriseMapImageLayerManager.build_cache) method. The first step is to define a [polygon]() object to define the specific area for the cache, and then define the scale level list for the cache.  The API for Python operation calls the ArcGIS for Server component's _CachingControllers_ geoprocessing server and runs the [_Manage Map Cache Tiles_](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/server/manage-map-server-cache-tiles.html?tabs=dialog) task in *RECREATE_ALL_TILES* mode. Click on the link to see full definitions and descriptions fo the various paramters and arguments (although the documentation shows the ArcGIS Pro dialog inputs, these parameters correpsond to parameters in API for Python).

This operation returns a Python dictionary, whose top level _status_ key may return a value of EXISTS, which in and of itself does not establish completion of the task. You must review the value for the *lodInfos* key and review each requested scales _tileCount_, _status_ and _isRunning_ values to verify the state of the operation. Unrequested scales can remain empty.

In [ ]:
# Defining the area of interest polygon
# The values of XMIN, YMIN, XMAX and YMAX correspond to the four endpoints of the polygon in map units
extent_poly = Polygon(
    iterable={
        "xmin": -13227632.572099,
        "ymin":3949348.377438,
        "xmax":-13004597.587167,
        "ymax":4095641.048130,
        "spatialReference": {"wkid":102100, "latestWkid": 3857}
    }
)

#Defining the scales at which the cache should be generated
lod=[18489297.737236001, 9244648.8686180003, 4622324.4343090001, 2311162.2171550002 ]

#Below are two different samples for generating the cache for the service within specific AOI or at the scale level

#Sample:1 - Generating tiles without specifing any area of interest
cache_job_scales = ent_mapimg_mgr.build_cache(
    levels=lod,
    area_of_interest=None
) 
cache_job_scales

# Sample:2 - Generating cache for the selected LOD levels and ared of Interest
cache_job_poly = ent_mapimg_mgr.build_cacje( levels=lod, area_of_interest=extent_poly) 

cache_job_poly

### 6. Managing the Cache
The API for Python provides the [update_tiles()](/python/latest/api-reference/arcgis.layers.toc.html#arcgis.layers.EnterpriseMapImageLayerManager.update_tiles) method for managing tiles after the cache is generated. This method calls the aforementioned _Manage Map Server Cache Tiles_ task on the ArcGIS Server's _CachingControllers_ geoprocessing server. The *update_mode* paramter allows for three argument values:
* *RECREATE_ALL_TILES* - replaces any existing tiles and creates any new ones for the *extent* or *area_of_interest*
* *RECREATE_EMPTY_TILES* - creates only any missing tiles in the *extent* or *area_of_interest*, leaves existing tiles unchanged
* *DELETE_TILES* - removes tiles from the cache in the *extent* or *area_of_interest*


#### 6a. Rebuild tiles at the configured scales
Using *RECREATE_ALL_TILES* for the *update_mode* argument regenerates tiles at the scales in the list used for the *levels* argument. Optionally you can define an *extent* or *area_of_interest* to focus the operation on a specific geographic location.

In [ ]:
#Defining Scales at the LOD levels:
lod=[18489297.737236001, 9244648.8686180003, 4622324.4343090001, 2311162.2171550002 ]

# Sample code for recreating tiles without providing an location
ent_mapimg_mgr.update_tiles
    levels=lod, 
    extent=None, 
    area_of_interest = None, 
    update_mode= "RECREATE_ALL_TILES"
)

#### 6b. Illustrating various options for empty tiles

Using *RECREATE_EMPTY_TILES* for the _update_mode_ argument will create empty tiles at the scales provided for the *levels* argument. 

Below we'll define a *polygon* and [*envelope*](https://developers.arcgis.com/python/latest/api-reference/arcgis.geometry.html#envelope) object. If using the *area_of_interest* argument, it  must be one of these objects. The *extent* argument can be a string or dictionary defining the extent.

Then we'll list 3 variations for how the method can be run:

In [ ]:
# Sample extent polygon
extent_poly = {
        "xmin": -13227632.572099,
        "ymin":3949348.377438,
        "xmax":-13004597.587167,
        "ymax":4095641.048130,
        "spatialReference": {"wkid":102100, "latestWkid": 3857}
}

# Sample Area of Interest variable for targeted deletion
aoi = Envelope(
    iterable= {
        "xmin": -118.995034,
        "ymin": 34.515780,
        "xmax": -118.932680,
        "ymax": 34.552597,
        "spatialReference": {
            "wkid": 4326
        }
    }
) 

In [ ]:
# Sample: 1 - Recreating Empty tiles at scale level across the whole extent of the layer
delete_lods = [18489297.737236001, 9244648.8686180003, 4622324.4343090001, 2311162.2171550002]

ent_mapimg_mgr.update_tiles(
    levels= delete_lods, 
    update_mode= "RECREATE_EMPTY_TILES"
)

# Sample: 2 - Recreating Empty tiles within a specific are if interest geometry
ent_mapimg_mgr.update_tiles(
    levels= delete_lods, 
    area_of_interest = aoi, 
    update_mode= "RECREATE_EMPTY_TILES"
)

# Sample: 3 - Recreating Empty tiles within a specific extent
ent_mapimg_mgr.update_tiles(
    levels= delete_lods, 
    extent= extent_poly, 
    update_mode= "RECREATE_EMPTY_TILES")

#### 6c. Delete tiles at selected scales
Use the *update_tiles()* operation with *DELETE_TILES* as the *update_mode* argument to remove tiles at the selected scales without executing a full-cache deletion. 
The code below has multiple examples for deleting tiles:

In [ ]:
#You can either delete the scales at the specific *levels* argument, or include an *area_of_interest* or *extent* argument to delete the tiles within a specific geographic area. 

#Below are two illlustrations of how *update_tiles()* can delete tiles for the cache, reusing previously defined variables:

#Sample: 1 - Deleting Tiles at specified scale levels across the whole extent of they layer. No geographic contsraints are used unless provided:
ent_mapimg_mgr.update_tiles(
    levels=delete_lods, 
    update_mode= "DELETE_TILES"
)

# Sample: 2 - Delete tiles within a specific areo of interest ad the desired scale *levels*:
ent_mapimg_mgr.update_tiles(
    levels=delete_lods,
    area_of_interest = aoi,
    update_mode= "DELETE_TILES"
)

#Sample: 3 - Delete tiles within a specific extent:
ent_mapimg_mgr.update_tiles(
    levels=delete_lods, 
    extent=extent_poly, 
    update_mode= "DELETE_TILES"
)

### 7. Delete Cache
Use the [delete_cache()](/python/latest/api-reference/arcgis.layers.toc.html#arcgis.layers.EnterpriseMapImageLayerManager.delete_cache) method to delete the cache schema along with all tiles. Further cache operations require recreating the schema and generating the needed tiles.

In [ ]:
#Delete cache operation does not require any inputs
ent_mapimg_mgr.delete_cache()